In [7]:
import os
import cv2
from tqdm import tqdm

VIDEO_DIR = "../../../data/videos"
OUTPUT_DIR = "../../data/frames"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Retrieve all video file paths
video_files = [os.path.join(VIDEO_DIR, f) for f in os.listdir(VIDEO_DIR) if f.endswith(('.mp4', '.avi'))]
video_files.sort()
print(f"Total videos found: {len(video_files)}")

def extract_frames(video_path, output_folder, num_frames=8):
    """
    Extract exactly 
um_frames uniformly sampled frames from the video.
    """
    vidcap = cv2.VideoCapture(video_path)
    if not vidcap.isOpened():
        print(f"Error: Cannot open video {video_path}")
        return
        
    total_frames = int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        vidcap.release()
        return

    # Calculate target frame indices for uniform sampling
    if total_frames <= num_frames:
        frame_indices = list(range(total_frames))
    else:
        interval = total_frames / num_frames
        frame_indices = [int(i * interval) for i in range(num_frames)]
        
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    video_out_dir = os.path.join(output_folder, video_name)
    os.makedirs(video_out_dir, exist_ok=True)
    
    current_frame = 0
    idx_pointer = 0
    frame_idx = 0
    
    while idx_pointer < len(frame_indices):
        success, image = vidcap.read()
        if not success:
            break
            
        target_frame = frame_indices[idx_pointer]
        if current_frame == target_frame:
            out_path = os.path.join(video_out_dir, f"frame_{frame_idx:04d}.jpg")
            cv2.imwrite(out_path, image)
            frame_idx += 1
            idx_pointer += 1
            
            # Handle edge cases where multiple target frames fall on the same index
            while idx_pointer < len(frame_indices) and frame_indices[idx_pointer] == target_frame:
                out_path = os.path.join(video_out_dir, f"frame_{frame_idx:04d}.jpg")
                cv2.imwrite(out_path, image)
                frame_idx += 1
                idx_pointer += 1
                
        current_frame += 1
        
    vidcap.release()

def process_batch(start_idx, end_idx):
    if start_idx >= len(video_files):
        print(f"Index {start_idx} is out of bounds. No more videos to process.")
        return
        
    end_idx = min(end_idx, len(video_files))
    batch = video_files[start_idx:end_idx]
    
    for video_path in tqdm(batch, desc=f"Processing batch {start_idx}-{end_idx}"):
        extract_frames(video_path, OUTPUT_DIR, num_frames=8)


Total videos found: 6285


In [8]:
process_batch(0, 1000)

Processing batch 0-1000:  10%|█         | 102/1000 [03:45<33:01,  2.21s/it] 


KeyboardInterrupt: 

In [ ]:
process_batch(1000, 2000)

In [ ]:
process_batch(2000, 3000)

In [ ]:
process_batch(3000, 4000)

In [ ]:
process_batch(4000, 5000)

In [ ]:
process_batch(5000, 6000)

In [ ]:
process_batch(6000, 6285)